# Pipeline Runner

Run the full poker ML pipeline with configurable sample size.

**IMPORTANT: Before running, update `NOTEBOOK_DIR` in cell 3 to match your Databricks workspace path!**

To find your path:
1. Right-click on any notebook in the Databricks workspace
2. Select "Copy path"
3. The path should look like: `/Workspace/Users/your.email@gmail.com/folder/notebooks`
4. Or for Repos: `/Repos/your.email@gmail.com/repo-name/notebooks`

**Pipeline Order:**
1. ~~01_DataIngestion~~ (commented out - run manually if needed)
2. 02_FeatureEngineering
3. 03_OpponentModeling
4. 04_AdvancedFeatures
5. 05_ProfitModel
6. 06_PolicyFeatures
7. 07_LabelCreation
8. 08_PlayerPerformance (identifies winning players)
9. ~~09_PolicyTraining~~ (commented out - run manually after validation)
10. 00_DataValidation (runs last to validate results)

**Note:** The original 08_PolicyTraining has been archived (`archive_08_PolicyTraining`) due to label leakage issues. The new approach:
- SP-8 identifies players with profit >= 5 BB/100 hands as "winners"
- SP-9 trains ONLY on winning player actions, using their actual decisions as labels

In [ ]:
# =============================================================================
# PIPELINE CONFIGURATION
# =============================================================================

# Sample size configuration
DEBUG_MODE = False          # True = sample data first, False = use all data (but still cap at MAX_ROWS)
SAMPLE_FRACTION = 0.01      # 1% of data (only used if DEBUG_MODE=True)
MAX_ROWS = 10000000         # Cap at this many rows (ALWAYS applied) - 10M to get ~1M after top 10% filter

# Training sample size for model-training notebooks (SP-03 and SP-05)
# These notebooks sample down to this size for sklearn training
# Other notebooks (SP-06 through SP-09) use the full MAX_ROWS
TRAINING_SAMPLE_SIZE = 300000  # 300k samples for model training

# =============================================================================
# What this means:
#
# DEBUG_MODE=True:
#   - First samples SAMPLE_FRACTION of data (e.g., 0.01 = 1%)
#   - Then caps at MAX_ROWS hands maximum
#   - Use for fast iteration/testing
#
# DEBUG_MODE=False:
#   - No sampling (starts with all data)
#   - Still caps at MAX_ROWS hands maximum
#   - Use for larger runs while still limiting compute
#
# TRAINING_SAMPLE_SIZE:
#   - SP-03 (Opponent Modeling) and SP-05 (Profit Model) sample to this size
#   - SP-06 through SP-09 use full dataset (need more data for top 10% filter)
#
# To process ENTIRE dataset: Set DEBUG_MODE=False and MAX_ROWS to a very large number
# =============================================================================

print("=" * 80)
print("PIPELINE CONFIGURATION")
print("=" * 80)
print(f"DEBUG_MODE: {DEBUG_MODE}")
print(f"MAX_ROWS: {MAX_ROWS:,} rows (for full pipeline)")
print(f"TRAINING_SAMPLE_SIZE: {TRAINING_SAMPLE_SIZE:,} rows (for SP-03 and SP-05)")
if DEBUG_MODE:
    print(f"SAMPLE_FRACTION: {SAMPLE_FRACTION} ({SAMPLE_FRACTION*100:.0f}% of data)")
    print(f"\n>>> Will sample {SAMPLE_FRACTION*100:.0f}% of data, then cap at {MAX_ROWS:,} rows")
else:
    print(f"\n>>> No sampling - will use up to {MAX_ROWS:,} rows from full dataset")
print(f">>> SP-03/SP-05 will sample down to {TRAINING_SAMPLE_SIZE:,} for model training")
print(f">>> SP-06 through SP-09 use full {MAX_ROWS:,} rows (need more for top 10% filter)")
print("=" * 80)

In [ ]:
# =============================================================================
# HELPER: Pass configuration to child notebooks via UC Volume (persists across Python restarts)
# Databricks Serverless doesn't allow custom Spark conf, so we use UC Volume
# =============================================================================
import json

# Use UC Volume path (persists across notebook runs and Python restarts)
CONFIG_PATH = '/Volumes/pokerml/default/data/pipeline_config.json'

config = {
    'debug_mode': DEBUG_MODE,
    'sample_fraction': SAMPLE_FRACTION,
    'max_rows': MAX_ROWS,
    'training_sample_size': TRAINING_SAMPLE_SIZE  # NEW: for SP-03 and SP-05
}

# Write config to UC Volume
with open(CONFIG_PATH, 'w') as f:
    json.dump(config, f)

print("Configuration saved to UC Volume")
print(f"  Path: {CONFIG_PATH}")
print(f"  Contents: {config}")

# Also try widgets (works for interactive runs)
try:
    dbutils.widgets.text("debug_mode", str(DEBUG_MODE), "Debug Mode")
    dbutils.widgets.text("sample_fraction", str(SAMPLE_FRACTION), "Sample Fraction")
    dbutils.widgets.text("max_rows", str(MAX_ROWS), "Max Rows")
    dbutils.widgets.text("training_sample_size", str(TRAINING_SAMPLE_SIZE), "Training Sample Size")
    print("\nWidgets also created (for interactive use)")
except Exception as e:
    print(f"\nWidgets not available: {e}")

In [ ]:
# =============================================================================
# NOTEBOOK PATHS
# =============================================================================
import time

# Databricks workspace path
NOTEBOOK_DIR = '/Workspace/Users/leo.lwakabamba@gmail.com/PokerML/files/notebooks'

# Pipeline notebooks in order
NOTEBOOKS = {
    # '01': f'{NOTEBOOK_DIR}/01_DataCollection',         # Commented out - run manually
    '02': f'{NOTEBOOK_DIR}/02_FeatureEngineering',
    '03': f'{NOTEBOOK_DIR}/03_OpponentModeling',
    '04': f'{NOTEBOOK_DIR}/04_AdvancedFeatures',
    '05': f'{NOTEBOOK_DIR}/05_ProfitModel',
    '06': f'{NOTEBOOK_DIR}/06_PolicyFeatures',
    '07': f'{NOTEBOOK_DIR}/07_LabelCreation',
    '08': f'{NOTEBOOK_DIR}/08_PlayerPerformance',        # NEW: Identifies winning players
    # '09': f'{NOTEBOOK_DIR}/09_PolicyTraining',         # Commented out - run after validation
    '00': f'{NOTEBOOK_DIR}/00_DataValidation',           # Validation runs last
}

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"\nNotebooks to run:")
for key, path in NOTEBOOKS.items():
    print(f"  SP-{key}: {path}")

In [ ]:
# =============================================================================
# RUN PIPELINE
# =============================================================================
results = {}
total_start = time.time()

print("\n" + "=" * 80)
print("STARTING PIPELINE RUN")
print("=" * 80)

# Timeout per notebook (in seconds)
# SP-03 and SP-05 train multiple models and can take several hours with large datasets
NOTEBOOK_TIMEOUT = 43200  # 12 hours per notebook

for key, notebook_path in NOTEBOOKS.items():
    print(f"\n{'='*60}")
    print(f"RUNNING: SP-{key} - {notebook_path.split('/')[-1]}")
    print(f"{'='*60}")
    
    start = time.time()
    try:
        # Run the notebook
        result = dbutils.notebook.run(notebook_path, timeout_seconds=NOTEBOOK_TIMEOUT)
        elapsed = time.time() - start
        
        results[key] = {
            'status': 'SUCCESS',
            'elapsed': elapsed,
            'result': result
        }
        print(f"\n✓ SP-{key} completed in {elapsed/60:.1f} minutes")
        
    except Exception as e:
        elapsed = time.time() - start
        results[key] = {
            'status': 'FAILED',
            'elapsed': elapsed,
            'error': str(e)
        }
        print(f"\n✗ SP-{key} FAILED after {elapsed/60:.1f} minutes")
        print(f"  Error: {e}")
        
        # Ask whether to continue or stop
        print("\n*** Pipeline stopped due to error ***")
        break

total_elapsed = time.time() - total_start
print(f"\n\nTotal pipeline time: {total_elapsed/60:.1f} minutes")

In [ ]:
# =============================================================================
# PIPELINE SUMMARY
# =============================================================================
print("\n" + "=" * 80)
print("PIPELINE SUMMARY")
print("=" * 80)

print(f"\nConfiguration:")
print(f"  DEBUG_MODE: {DEBUG_MODE}")
if DEBUG_MODE:
    print(f"  SAMPLE_FRACTION: {SAMPLE_FRACTION*100:.0f}%")
    print(f"  MAX_ROWS: {MAX_ROWS:,}")

print(f"\nResults:")
print(f"{'Notebook':<30} {'Status':<10} {'Time':<15}")
print("-" * 55)

success_count = 0
fail_count = 0

for key, res in results.items():
    notebook_name = NOTEBOOKS[key].split('/')[-1]
    status = res['status']
    elapsed = f"{res['elapsed']:.1f}s"
    
    if status == 'SUCCESS':
        success_count += 1
        print(f"{notebook_name:<30} {'✓ ' + status:<10} {elapsed:<15}")
    else:
        fail_count += 1
        print(f"{notebook_name:<30} {'✗ ' + status:<10} {elapsed:<15}")

print("-" * 55)
print(f"Total: {success_count} succeeded, {fail_count} failed")
print(f"Total time: {total_elapsed/60:.1f} minutes")

if fail_count == 0:
    print("\n" + "=" * 80)
    print("✓ PIPELINE COMPLETED SUCCESSFULLY")
    print("=" * 80)
    print("\nNext steps:")
    print("1. Review validation results above (from 00_DataValidation)")
    print("2. Review player performance visualizations from 08_PlayerPerformance")
    print("3. If validation passes, run 09_PolicyTraining manually")
else:
    print("\n" + "=" * 80)
    print("✗ PIPELINE FAILED")
    print("=" * 80)
    print("\nCheck the error messages above and fix before re-running")

---

## Manual Notebook Runs

If you need to run individual notebooks manually, uncomment and run the cells below:

In [ ]:
# # Uncomment to run 01_DataIngestion manually
# # This is typically only needed once to ingest raw data
# 
# # dbutils.notebook.run(f'{NOTEBOOK_DIR}/01_DataIngestion', timeout_seconds=3600)

In [ ]:
# # Uncomment to run 09_PolicyTraining manually
# # Run this after validation passes
# # This trains on winning player actions only (no label leakage)
# 
# # dbutils.notebook.run(f'{NOTEBOOK_DIR}/09_PolicyTraining', timeout_seconds=3600)

In [ ]:
# # Uncomment to run just the validation notebook
# 
# # dbutils.notebook.run(f'{NOTEBOOK_DIR}/00_DataValidation', timeout_seconds=600)